# Qwen3-1.7B: first 300-example SFT experiment

Ordinary **LoRA is the default**. QLoRA is an explicit fallback only after a recorded LoRA out-of-memory failure or failure to meet this experiment's 15% free-memory target. Nothing changes modes automatically. The CUDA combination and output quality remain unverified until actual execution.

All 300 approved `sft-300-v2` rows train for one epoch, with thinking and packing disabled and loss on the final completion only. Independent model audit passed; human review remains pending. Preserve existing baseline responses. No held-out inputs are included.

Sequence: configure → install → Drive → verify kit/data → all-row mask audit → two-update GPU smoke with longest/history → discard live smoke state → fresh full training → adapter reload/inference → export. Distributed gates remain `RUN_SMOKE=False` and `RUN_TRAINING=False`.


## 1. Review the fixed experiment settings

Select **Runtime → Change runtime type → GPU**. Use a fresh `RUN_NAME` for every smoke/mode attempt. Ordinary LoRA uses the unquantized pinned base in BF16 when supported, otherwise FP16 mixed precision, and FP32 trainable adapters. Both modes freeze the base and use gradient checkpointing. QLoRA alone installs bitsandbytes and uses NF4 double quantization.

The smoke performs two optimizer updates with the same microbatch 1 / accumulation 8 as full training, using the full run's first eight seeded shuffled rows, followed by eight alternating longest/history rows. The full run freshly initializes the base/adapter with the original seed. Resume is separate: set an exact existing checkpoint path only after an interruption; complete checkpoint state and immutable run metadata must match.

FP16 uses an explicitly recorded initial GradScaler scale of 1024. A controlled T4 replay found overflow at the default 65536 while 1024 and 256 produced finite gradients on the same eight rows without changing weights. The helper sets the scaler through its public state API before training; checkpoint resume restores its saved scaler afterward. Finite-gradient checks stay strict, and skipped optimizer updates fail the run. BF16 uses no scaler.


In [ ]:
# Run this first in a restarted kernel, before any torch/dependency import.
import os, sys
assert 'torch' not in sys.modules, 'Restart the runtime, then run this configuration cell first'
assert 'PYTORCH_ALLOC_CONF' not in os.environ, 'Conflicting allocator setting; inspect before proceeding'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
ALLOCATOR_CONFIG = os.environ['PYTORCH_CUDA_ALLOC_CONF']

from pathlib import Path
import json

RUN_NAME = 'qwen-sft300-v2-lora-001'
TRAINING_MODE = 'lora'   # 'lora' first; 'qlora' only after documented memory failure.
MODE_REASON = 'Ordinary LoRA is the preferred initial experiment.'
LORA_FAILURE_REPORT = None  # QLoRA: absolute Drive path to prior LoRA smoke-report.json.
RESUME_CHECKPOINT = None    # Exact .../full/checkpoints/checkpoint-N; never True/automatic latest.
RUN_SMOKE = False
RUN_TRAINING = False
DOWNLOAD_ADAPTER = False
BASELINE_REFERENCE = 'evaluation-sft-v1/baseline-reference/manifest.json sha256:9215dc370204863e1ff58383d6e6083e709a883af54e74f80f9d7a54c82b89b6'
MAX_LENGTH = 2048
SEED = 42
LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LEARNING_RATE = 1e-4
EPOCHS = 1.0
MICROBATCH = 1
GRADIENT_ACCUMULATION = 8
FP16_INITIAL_SCALE = 1024.0  # Measured T4 fix; used identically by smoke/full.
assert '/' not in RUN_NAME and RUN_NAME not in ('', '.', '..')
assert TRAINING_MODE in ('lora', 'qlora') and MODE_REASON.strip()
assert TRAINING_MODE == 'lora' or LORA_FAILURE_REPORT, 'QLoRA needs saved LoRA memory diagnostics'
assert RESUME_CHECKPOINT is None or (RUN_TRAINING and not RUN_SMOKE), 'Resume uses the original smoke record'
print({'mode': TRAINING_MODE, 'RUN_SMOKE': RUN_SMOKE, 'RUN_TRAINING': RUN_TRAINING})


## 2. Install the pinned libraries

Keep Colab's CUDA PyTorch (>=2.4). Ordinary LoRA does not require bitsandbytes. The explicit QLoRA branch adds its pin. No TRL or datasets dependency is needed. If packages were already imported, restart the runtime after installation before continuing.

Record global `pip check` conflicts for diagnosis; Colab may contain unrelated conflicting preinstalled packages. Required package versions and imports are checked directly. A dependency/precision/masking error must be fixed, and is never grounds for switching to QLoRA.

PEFT also probes optional torchao during ordinary LoRA attachment. The cell records any preinstalled version and removes only incompatible torchao <0.16.0 before imports; this experiment does not need torchao. It clears availability caches and requests a runtime restart if torchao had already been imported. A tiny CPU-only adapter attachment then exercises PEFT dispatch before any base-model download or GPU optimization. Its result and package action are saved in dependency diagnostics.


In [ ]:
import subprocess, sys, importlib, importlib.metadata
PINS = ['transformers==4.57.6', 'peft==0.21.2', 'accelerate==1.15.0',
        'tokenizers==0.22.2', 'huggingface_hub==0.36.2', 'safetensors==0.8.0']
if TRAINING_MODE == 'qlora':
    PINS.append('bitsandbytes==0.49.2')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *PINS], check=True)
# PEFT 0.21.2 probes optional torchao during even ordinary LoRA dispatch.
# Colab may preinstall torchao 0.10.0; remove only incompatible optional versions.
from packaging.version import Version

def prepare_optional_torchao():
    try:
        before = importlib.metadata.version('torchao')
    except importlib.metadata.PackageNotFoundError:
        before = None
    record = {'before_version': before, 'minimum_if_present': '0.16.0',
              'action': 'absent' if before is None else 'retained_compatible',
              'restart_required': False, 'peft_probe_cache_cleared': False}
    if before is not None and Version(before) < Version('0.16.0'):
        record['restart_required'] = any(name == 'torchao' or name.startswith('torchao.') for name in sys.modules)
        subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '--yes', 'torchao'], check=True)
        record['action'] = 'removed_incompatible_optional_package'
        importlib.invalidate_caches()
        loaded_probes = sys.modules.get('peft.import_utils')
        if loaded_probes is not None:
            loaded_probes.is_torchao_available.cache_clear()
            record['peft_probe_cache_cleared'] = True
    try:
        record['after_version'] = importlib.metadata.version('torchao')
    except importlib.metadata.PackageNotFoundError:
        record['after_version'] = None
    assert record['after_version'] is None or Version(record['after_version']) >= Version('0.16.0'), 'Incompatible torchao remains'
    return record

TORCHAO_PREFLIGHT = prepare_optional_torchao()
pip_check = subprocess.run([sys.executable, '-m', 'pip', 'check'], text=True, capture_output=True)
PIP_CHECK_DIAGNOSTICS = {'returncode': pip_check.returncode, 'stdout': pip_check.stdout, 'stderr': pip_check.stderr,
                         'optional_torchao_preflight': TORCHAO_PREFLIGHT}
assert not TORCHAO_PREFLIGHT['restart_required'], 'Removed previously imported torchao: restart runtime and rerun setup'
for pin in PINS:
    name, expected = pin.split('==')
    assert importlib.metadata.version(name) == expected, f'{name}: restart session after install'
for pin in PINS:
    name, expected = pin.split('==')
    module = importlib.import_module(name)
    assert module.__version__ == expected, f'{name}: imported old version; restart runtime'
import torch, transformers, peft, accelerate
if TRAINING_MODE == 'qlora':
    import bitsandbytes
# Exercise PEFT's real CPU adapter dispatch before downloading the base model.
dispatch_probe = peft.get_peft_model(torch.nn.Sequential(torch.nn.Linear(2, 2, device='cpu')),
    peft.LoraConfig(r=2, lora_alpha=4, target_modules=['0']))
assert any('lora_' in name and parameter.requires_grad for name, parameter in dispatch_probe.named_parameters())
PIP_CHECK_DIAGNOSTICS['peft_cpu_adapter_dispatch'] = {'passed': True, 'model_downloaded': False, 'optimizer_steps': 0}
del dispatch_probe
print(PIP_CHECK_DIAGNOSTICS)
print('Required imports passed. GPU execution remains a separate check.')


## 3. Mount Drive and select a durable run directory

The user must complete Drive authorization. New attempts require a fresh experiment root, preserving every previous failure and result. An explicit resume may reuse only its original run root, without overwriting setup records. The helper verifies checkpoint ownership, configuration, mode, source/template hashes, dependency versions, precision, optimizer, scheduler, RNG and (FP16) scaler state before loading it.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/Reading-Model/sft')
EXPERIMENT = DRIVE_ROOT / RUN_NAME
if RESUME_CHECKPOINT is None:
    assert not EXPERIMENT.exists(), 'Use a new RUN_NAME; previous evidence must be preserved'
    EXPERIMENT.mkdir(parents=True)
else:
    assert EXPERIMENT.is_dir()
    checkpoint = Path(RESUME_CHECKPOINT).resolve()
    assert checkpoint.parent == (EXPERIMENT / 'full' / 'checkpoints').resolve()
print('Experiment folder:', EXPERIMENT)


## 4. Upload the minimal SFT kit

Upload `reading-sft-colab-v2.zip` from this project. It contains only the training helper, pinned requirements, approved SFT data, exact source manifest, and file hashes. Review reasoning, audit report contents, baseline runs, credentials, model weights, and held-out data are excluded. The exact source manifest still binds the independent audit outcome and report hashes. The notebook pins the whole ZIP hash before extracting anything.


In [ ]:
from google.colab import files
import hashlib, io, zipfile, shutil
uploaded = files.upload()
assert len(uploaded) == 1 and 'reading-sft-colab-v2.zip' in uploaded, 'Upload only reading-sft-colab-v2.zip'
archive_bytes = uploaded['reading-sft-colab-v2.zip']
EXPECTED_KIT_SHA256 = '84df1923ca4101eaefb4548687c27ff5c5737405eb388bbe11b935ef50bec939'
assert hashlib.sha256(archive_bytes).hexdigest() == EXPECTED_KIT_SHA256, 'Wrong or changed upload kit'
KIT = Path('/content/reading-sft-colab-v2')
assert not KIT.exists(), 'Kit folder exists; restart the runtime or inspect/remove that folder before rerunning upload'
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    expected_names = {'sft_train.py', 'requirements-sft-colab.txt', 'data/sft.jsonl', 'data/manifest.json', 'kit-manifest.json'}
    assert set(archive.namelist()) == expected_names and len(archive.namelist()) == len(expected_names)
    for entry in archive.infolist():
        target = (KIT / entry.filename).resolve()
        assert target.is_relative_to(KIT.resolve()) and not entry.is_dir()
        assert (entry.external_attr >> 16) & 0o170000 != 0o120000, 'Symlinks are forbidden'
    archive.extractall(KIT)
kit_manifest = json.loads((KIT / 'kit-manifest.json').read_text())
for name, expected in kit_manifest['files'].items():
    assert hashlib.sha256((KIT / name).read_bytes()).hexdigest() == expected, name
sys.path.insert(0, str(KIT))
import sft_train as training
print('Exact kit verified:', EXPECTED_KIT_SHA256)


## 5. Verify the approved data and record the runtime

The helper pins the finalized v2 manifest and SFT SHA-256 hashes, verifies 300 rows, and requires the manifest's independent audit outcome to be 300 pass with zero open findings. This is model review, with human review pending. The trainer sees only prompt/completion text. No test or archived benchmark is available in the kit.


In [ ]:
data, source_manifest = training.verify_data(KIT / 'data')
config = training.TrainConfig(mode=TRAINING_MODE, seed=SEED, max_length=MAX_LENGTH, rank=LORA_RANK,
    alpha=LORA_ALPHA, dropout=LORA_DROPOUT, learning_rate=LEARNING_RATE,
    epochs=EPOCHS, microbatch=MICROBATCH, gradient_accumulation=GRADIENT_ACCUMULATION,
    fp16_initial_scale=FP16_INITIAL_SCALE)
environment = training.cuda_environment(config.mode)
fallback_evidence = None
if TRAINING_MODE == 'qlora':
    failure_path = Path(LORA_FAILURE_REPORT)
    prior = json.loads(failure_path.read_text())
    assert prior['mode'] == 'lora' and not prior['passed']
    assert prior.get('failure_kind') in ('out_of_memory', 'insufficient_headroom'), 'Fix non-memory failures first'
    assert prior['identity']['config'] == {**training.asdict(config), 'mode': 'lora'}
    fallback_evidence = {'path': str(failure_path), 'sha256': training.sha256(failure_path),
                         'failure_kind': prior['failure_kind']}
RUN_CONTEXT = {'kit_sha256': EXPECTED_KIT_SHA256, 'baseline_reference': BASELINE_REFERENCE,
               'allocator_config': ALLOCATOR_CONFIG, 'mode_reason': MODE_REASON, 'lora_failure_evidence': fallback_evidence}
setup = {'training': training.asdict(config), 'model': training.MODEL_ID,
         'revision': training.REVISION, 'context': RUN_CONTEXT,
         'source_manifest_sha256': training.MANIFEST_SHA256}
if RESUME_CHECKPOINT is None:
    training.write_json(EXPERIMENT / 'environment.json', environment)
    training.write_json(EXPERIMENT / 'pip-check.json', PIP_CHECK_DIAGNOSTICS)
    training.write_json(EXPERIMENT / 'config.json', setup)
else:
    assert json.loads((EXPERIMENT / 'config.json').read_text()) == setup, 'Resume setup differs'
print(json.dumps(environment, indent=2))
print('Approved dataset:', source_manifest['version'], source_manifest['audit_status'])


## 6. Tokenize all 300 examples without truncation

This downloads only the tokenizer at the pinned Qwen revision. Apply the native template with `enable_thinking=False`; its inference prefix contains an empty thinking block. Only the final response and end-of-message suffix become labels. System text, user text, historical assistant responses, and the empty thinking prefix are all `-100` (ignored by the causal-LM loss).

Current offline measurements are 1,737–1,825 full tokens. A 2,048 limit fits; a 1,024 default would truncate every example. The helper fails on any overlength row or changed template boundary. Do not lower the limit by allowing silent truncation.


In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(training.MODEL_ID, revision=training.REVISION)
tokenizer.padding_side = 'right'
assert tokenizer.pad_token_id is not None
features = training.tokenize_rows(data, tokenizer, max_length=config.max_length)
assert len(features) == 300
print({'min_tokens': min(len(f['input_ids']) for f in features),
       'max_tokens': max(len(f['input_ids']) for f in features),
       'historical_assistant_turns': sum(m['role'] == 'assistant' for row in data for m in row['prompt'])})


## 7. Audit the actual training collator on every row

This is the **same collator passed to Trainer**, rather than a separate estimate. All 300 padded examples are checked: nonempty final labels, exact decoded targets/terminators, prompt/history excluded, padding excluded, and model inputs containing only IDs, attention masks, and labels. EOS is retained even if a runtime uses the same token ID for EOS and padding. No packing or template-based all-assistant mask is used.

The helper's all-row collator tests pass locally with the cached tokenizer. GPU forward/backward and save/reload are still separate checks below.


In [ ]:
collator = training.CompletionCollator(tokenizer.pad_token_id, max_length=config.max_length)
mask_report = training.verify_collator(features, collator, tokenizer)
assert mask_report['supervised_tokens'] == 4683, 'Unexpected change to tokenization'
if RESUME_CHECKPOINT is None:
    training.write_json(EXPERIMENT / 'actual-collator-report.json', mask_report)
else:
    assert json.loads((EXPERIMENT / 'actual-collator-report.json').read_text()) == mask_report
print(json.dumps(mask_report, indent=2))
for feature in training.representative_smoke_features(features, config)[8:10]:
    batch = collator([feature])
    labels = batch['labels'][0].tolist()
    print(feature['id'], tokenizer.decode([x for x in labels if x != -100], skip_special_tokens=False))


## 8. Optional GPU smoke in the explicitly selected mode

Enable `RUN_SMOKE` to download weights. The all-300 collator audit repeats, then the first accumulated update uses the exact first eight seeded shuffled full-training rows from fresh weights; the second uses alternating longest/history rows. Actual model batches must retain their exact masks; loss and nonzero adapter gradients must be finite, the base frozen, and adapter weights changed. Both optimizer updates must actually apply; a skipped update stops the run.

The smoke saves/reloads the adapter against the same pinned base and verifies exact weight hashes, finite forward loss, and generated tokens. It records GPU peak allocated/reserved memory and sampled driver free memory through training/reload. Passing requires at least 15% free memory by conservative measurements; this is an operational target for this experiment. No fit is assumed.

A failure is recorded before raising. For LoRA OOM or insufficient margin only: release models/optimizers (the helper cleans up), restart the runtime if needed, choose a fresh run name, set `TRAINING_MODE='qlora'`, give the LoRA report path and reason, and repeat the full smoke. Never switch for unrelated errors. If QLoRA also fails, stop with both diagnostic reports.


In [ ]:
smoke_report = None
if RESUME_CHECKPOINT is not None:
    smoke_report = json.loads((EXPERIMENT / 'smoke' / 'smoke-report.json').read_text())
    assert smoke_report['identity'] == training.run_identity(config, tokenizer, RUN_CONTEXT)
    assert smoke_report['passed'] and smoke_report['headroom_passed']
elif RUN_SMOKE:
    smoke_report = training.smoke_test(features, collator, tokenizer, config,
        EXPERIMENT / 'smoke', context=RUN_CONTEXT)
    print(json.dumps(smoke_report, indent=2))
else:
    print('Smoke skipped. No model loaded or optimization started.')


## 9. Review smoke evidence before full training

A passing smoke must show two applied optimizer updates with zero skips, actual representative batches, finite nonzero adapter gradients, frozen base, changed adapter weights, successful save/reload/inference and 15% memory headroom. The smoke's live models and optimizers have been discarded. Its diagnostic adapter remains in Drive; the full run never trains from it.

The full helper checks identical mode/config/data/template/source/context and precision/dependencies, then freshly initializes with the same seed and verifies the original pre-smoke fingerprint. Explicit interruption resume instead restores the full run's matching optimizer checkpoint. Independent code review is required before enabling full training.


In [ ]:
if smoke_report is not None:
    assert smoke_report['passed'] and smoke_report['headroom_passed'], 'Memory target not met'
    assert smoke_report['save_reload_weights_equal'] and smoke_report['inference_verified']
    assert smoke_report['smoke_weights_discarded_before_full_run']
    print('Smoke passed in mode', config.mode, '; review evidence and code before full training.')
else:
    print('Full training blocked pending passing smoke.')


## 10. Optional full one-epoch run on all 300 rows

Enable `RUN_TRAINING` after a passing smoke and independent code review. Microbatch 1 / accumulation 8 yields 38 optimizer updates. The helper audits actual training batch labels and row coverage, checks finite loss/gradients, logs memory and saves complete checkpoints every 10 updates. It freshly initializes the base/adapter unless resuming an explicitly verified full-run checkpoint.

Drive receives adapter, tokenizer, mode/configuration, dataset/template/source hashes, environment versions, logs and run record. Completion is recorded only after releasing training state and successfully reloading the final adapter, verifying its weights and running generation. A lower training loss is an engineering signal, not proof of better behavior. No baseline/held-out inference runs here.

After smoke passes, set `RUN_TRAINING = True` in a separate cell and run the full-training cell. Do not rerun the initial configuration cell: it intentionally requires a fresh kernel before torch is imported.


In [ ]:
training_record = None
if RUN_TRAINING:
    assert smoke_report is not None and smoke_report['passed'], 'Pass RUN_SMOKE first'
    training_record = training.full_train(features, collator, tokenizer, config,
        EXPERIMENT / 'full', smoke_report, source_manifest, context=RUN_CONTEXT,
        resume_checkpoint=RESUME_CHECKPOINT)
    assert training_record['status'] == 'training_completed'
    print(json.dumps(training_record, indent=2))
else:
    print('Full training skipped. RUN_TRAINING is False.')


## 11. Export the verified adapter and reproducibility records

The usable result is a PEFT adapter with its pinned base revision. The adapter folder and run record both record training mode. The export includes successful final reload evidence, smoke/mask reports, environment, setup and dependency diagnostics; checkpoints and the diagnostic smoke adapter stay in Drive. Optional downloading does not publish the model.


In [ ]:
if training_record is not None:
    assert training_record['status'] == 'training_completed' and training_record['final_reload']['inference_verified']
    bundle = EXPERIMENT / 'adapter-export.zip'
    assert not bundle.exists(), 'Export already exists; inspect rather than overwrite'
    with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted((EXPERIMENT / 'full' / 'adapter').iterdir()):
            if path.is_file():
                archive.write(path, 'adapter/' + path.name)
        for name in ['run-record.json', 'environment-freeze.txt', 'trainer-log.json']:
            archive.write(EXPERIMENT / 'full' / name, name)
        for name in ['actual-collator-report.json', 'environment.json', 'config.json', 'pip-check.json']:
            archive.write(EXPERIMENT / name, name)
        archive.write(EXPERIMENT / 'smoke' / 'smoke-report.json', 'smoke-report.json')
    print('Saved:', bundle, 'SHA256:', training.sha256(bundle))
    if DOWNLOAD_ADAPTER:
        files.download(str(bundle))
else:
    print('No verified full adapter in this session; nothing exported.')


## 12. Compare development responses with the existing saved baseline

The baseline is already saved; do not rerun it just to complete this notebook. Preserve its result export and settings as the comparison reference. Generate **adapter-only** responses for the same development inputs with matching prompt A/B, Qwen revision, thinking disabled, precision/device as feasible, and decoding settings (the local engine uses temperature 0.7, top-p 0.8, top-k 20, maximum 128 new tokens). Record any hardware or revision mismatch rather than assuming comparability.

The existing project CLI can load the extracted adapter on the full project, for example:

```sh
python run_baseline.py --backend qwen --adapter /absolute/path/to/adapter --split dev --out runs/qwen-sft300-v2-dev
```

This invokes the adapter variant only. The evaluation kit/CLI is deliberately absent from this minimal training upload. `QwenEngine` now pins model and tokenizer loading to the training revision and rejects adapters with a missing or different revision. Verify actual precision/device/template/decoding settings before a strict comparison. Put the adapter's local absolute path in `models.json` → `qwen_sft.adapter` when ready to use the existing UI; do not change baseline records.

Blind-score reading eligibility, length, rhythm, meaningful answers, and regressions against saved baseline responses. Do not judge generalization from training examples. Training choices and checkpoint selection should use development material. No held-out/test files are part of this notebook or kit. Human review and a later properly frozen evaluation remain outstanding.

### Sources and verification limits

The implementation follows the [Transformers Trainer API](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/trainer), [PEFT quantization guide](https://huggingface.co/docs/peft/developer_guides/quantization), and [pinned Qwen model card](https://huggingface.co/Qwen/Qwen3-1.7B/blob/70d244cc86ccca08cf5af4e1e306ecf908b1ad5e/README.md). Release existence was checked on PyPI for [Transformers](https://pypi.org/project/transformers/4.57.6/), [PEFT](https://pypi.org/project/peft/0.21.2/), [Accelerate](https://pypi.org/project/accelerate/1.15.0/), and [bitsandbytes](https://pypi.org/project/bitsandbytes/0.49.2/). These checks do not establish CUDA compatibility. No Colab install, GPU load, training step, or adapter quality improvement has been completed merely by preparing this notebook.
